#Tests

In [ ]:
!pip install pytest torch transformers Pillow numpy scikit-learn pandas accelerate scipy opencv-python -q

In [ ]:
%%writefile test_deepfake_detector.py

import pytest
import numpy as np
import torch
from PIL import Image
import pandas as pd
import os
import tempfile
import warnings

# ===========================================================================
# Fixtures – shared setup objects
# ===========================================================================

@pytest.fixture(scope="module")
def processor_and_model():
    """
    Loads the real InstructBLIP processor and model once for the entire module.

    IMPORTANT (Colab memory note):
    instructblip-vicuna-7b has ~7B parameters. Loading in float32 on CPU
    requires ~28GB RAM, which exceeds free Colab's ~12.6GB and will crash
    with OOM. On GPU runtimes, load in float16 with device_map="auto".
    On CPU-only runtimes, this fixture will be slow/heavy.

    Covers: InstructBLIPModelLoad
    """
    from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

    proc = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")

    # 'query_tokens' is a Parameter (not a submodule), so accelerate's
    # device_map="auto" logic emits a harmless UserWarning saying it can't
    # find a submodule by that name. The parameter still loads and is
    # placed correctly (verified by test_model_load / test_embedding_
    # extraction_shape passing). We filter only this exact, known-benign
    # warning so test output stays clean without masking real warnings.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r".*device_map keys do not match any submodules.*query_tokens.*",
            category=UserWarning,
        )
        if torch.cuda.is_available():
            mdl = InstructBlipForConditionalGeneration.from_pretrained(
                "Salesforce/instructblip-vicuna-7b",
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            mdl = InstructBlipForConditionalGeneration.from_pretrained(
                "Salesforce/instructblip-vicuna-7b",
                torch_dtype=torch.float32
            )

    mdl.eval()
    for param in mdl.parameters():
        param.requires_grad = False
    return proc, mdl


def _get_device(mdl):
    return next(mdl.vision_model.parameters()).device


def _get_dtype(mdl):
    return next(mdl.vision_model.parameters()).dtype


# ---------------------------------------------------------------------------
# Probabilistic inference helpers
# ---------------------------------------------------------------------------
# Normalization algorithm (Phase B, §3.2):
#   1. Define Yes/No token variants and de-duplicate their IDs.
#   2. Take MAX logit within each subset (handles tokenizer variation).
#   3. Apply Softmax strictly over the {Yes, No} pair — NOT over the full
#      vocabulary — so probabilities sum to 1 without noise from irrelevant
#      tokens (e.g. "maybe", "car").
#
#   P(Fake) = softmax([yes_logit, no_logit])[1]   (index 1 == "No" == Fake)
# ---------------------------------------------------------------------------

YES_TOKENS = ["Yes", "yes", " Yes", " yes"]
NO_TOKENS  = ["No",  "no",  " No",  " no"]


def get_yes_no_token_ids(tokenizer, device):
    """Encode YES_TOKENS/NO_TOKENS to de-duplicated token ID tensors."""
    yes_ids = list({tokenizer.encode(t, add_special_tokens=False)[-1] for t in YES_TOKENS})
    no_ids  = list({tokenizer.encode(t, add_special_tokens=False)[-1] for t in NO_TOKENS})

    overlap = set(yes_ids) & set(no_ids)
    if overlap:
        raise ValueError(
            f"Yes/No token sets overlap at IDs {overlap} — "
            f"classification is undefined for this tokenizer."
        )
    return (torch.tensor(yes_ids, device=device),
            torch.tensor(no_ids, device=device))


def normalize_to_fake_probability(yes_logit, no_logit):
    """
    P(Fake) via Softmax over the {Yes, No} subset only (Phase B, §3.2).

    Steps:
      - Stack the two scalar logits into a (batch, 2) tensor.
      - Apply Softmax over dim=-1 so probabilities sum to 1.
      - Return index 1 (the "No" / Fake probability).

    Why Softmax over the subset (not full vocabulary):
      Normalizing over all ~32,000 vocab tokens would let noise from
      irrelevant tokens dilute the score. Restricting to {Yes, No} ensures
      the result faithfully reflects the model's relative confidence between
      the two classes only.

    Why Softmax (not raw division P_no / (P_yes + P_no)):
      Raw division requires both logits to be positive; if either is
      negative the denominator can be ≤ 0, producing values outside [0, 1].
      Softmax uses exp(), which is always positive, so the result is always
      a valid probability regardless of logit sign.
    """
    pair_logits = torch.stack([yes_logit, no_logit], dim=-1)
    pair_probs  = torch.softmax(pair_logits, dim=-1)
    return pair_probs[:, 1]  # index 1 == "No" == Fake


def _run_probabilistic_inference(mdl, proc, image):
    """
    Single-image inference matching the batch logic used in all evaluation
    notebooks:
      generate(max_new_tokens=1) → MAX over de-duped Yes/No IDs → Softmax.

    Returns: fake_prob (float in [0, 1])
    """
    device = _get_device(mdl)
    dtype  = _get_dtype(mdl)

    inputs = proc(images=image, text="is this photo real?", return_tensors="pt")
    inputs = {k: (v.to(device, dtype=dtype) if k == 'pixel_values' else v.to(device))
              for k, v in inputs.items()}

    yes_ids, no_ids = get_yes_no_token_ids(proc.tokenizer, device)

    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=1,
            return_dict_in_generate=True,
            output_scores=True
        )

    first_token_logits = outputs.scores[0]  # (1, vocab_size)
    yes_logit = first_token_logits[:, yes_ids].max(dim=-1).values
    no_logit  = first_token_logits[:, no_ids].max(dim=-1).values

    fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
    return fake_prob.item()


@pytest.fixture
def dummy_rgb_image():
    return Image.new('RGB', (256, 256), color=(128, 64, 32))


@pytest.fixture
def dummy_face_image():
    return Image.new('RGB', (224, 224), color=(200, 180, 160))


# ===========================================================================
# 1. InstructBLIP Module Tests
# ===========================================================================

class TestInstructBLIPModule:

    # -----------------------------------------------------------------------
    # InstructBLIPModelLoad
    # -----------------------------------------------------------------------
    def test_model_load(self, processor_and_model):
        """
        InstructBLIPModelLoad:
        Verify that the processor and model load into memory without errors,
        and that the Q-Former and vision model sub-modules exist.
        """
        proc, mdl = processor_and_model
        assert proc is not None
        assert mdl  is not None
        assert hasattr(mdl, 'qformer'),      "Model must have a qformer component"
        assert hasattr(mdl, 'vision_model'), "Model must have a vision_model component"
        assert hasattr(mdl, 'query_tokens'), "Model must have query_tokens"

    # -----------------------------------------------------------------------
    # InstructBLIPImagePrep
    # -----------------------------------------------------------------------
    def test_image_preparation(self, processor_and_model, dummy_face_image):
        """
        InstructBLIPImagePrep:
        Verify that the processor converts a PIL image into the tensor format
        expected by the model.
        """
        proc, _ = processor_and_model
        inputs = proc(
            images=dummy_face_image,
            text="is this photo real?",
            return_tensors="pt"
        )
        assert 'pixel_values'           in inputs
        assert 'input_ids'              in inputs
        assert 'qformer_input_ids'      in inputs
        assert 'qformer_attention_mask' in inputs
        assert inputs['pixel_values'].ndim == 4,     "pixel_values must be 4-D: (B, C, H, W)"
        assert inputs['pixel_values'].shape[1] == 3, "pixel_values must have 3 channels (RGB)"

    # -----------------------------------------------------------------------
    # InstructBLIPGetText
    # -----------------------------------------------------------------------
    def test_get_text_output(self, processor_and_model, dummy_face_image):
        """
        InstructBLIPGetText:
        Verify that the model returns a non-empty string when prompted.
        We do NOT assert Yes/No here because the model's response to a
        synthetic solid-color image is undefined — testing for a specific
        answer would be fragile and unrelated to pipeline correctness.
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)
        dtype  = _get_dtype(mdl)

        inputs = proc(images=dummy_face_image, text="is this photo real?", return_tensors="pt")
        inputs = {k: (v.to(device, dtype=dtype) if k == 'pixel_values' else v.to(device))
                  for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = mdl.generate(**inputs, max_new_tokens=5)

        answer = proc.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        assert len(answer) > 0, "Model must return a non-empty text answer"

    # -----------------------------------------------------------------------
    # InstructBLIPMathCheck – normalization formula
    # -----------------------------------------------------------------------
    def test_normalization_formula_basic(self):
        """
        InstructBLIPMathCheck:
        Verify normalize_to_fake_probability returns P_fake in [0, 1]
        and P_real + P_fake == 1.

        Algorithm (Phase B §3.2): Softmax over {yes_logit, no_logit} subset.
        Index 0 = P(Real), index 1 = P(Fake).
        """
        yes_logit = torch.tensor([0.3])
        no_logit  = torch.tensor([0.7])

        fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
        real_prob = 1.0 - fake_prob

        assert 0.0 <= fake_prob.item() <= 1.0
        assert abs((fake_prob + real_prob).item() - 1.0) < 1e-5

    def test_normalization_handles_negative_logits(self):
        """
        Verify that normalize_to_fake_probability stays within [0, 1] even
        when raw logits are negative.

        This is a key advantage of the Softmax approach over raw division
        (P_no / (P_yes + P_no)): if either logit is negative, raw division
        can produce a denominator ≤ 0 and a result outside [0, 1].
        Softmax uses exp(), which is always positive, so the result is
        always a valid probability.
        """
        yes_logit = torch.tensor([-2.0])
        no_logit  = torch.tensor([-5.0])

        fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
        assert 0.0 <= fake_prob.item() <= 1.0, \
            f"P_fake must stay in [0,1] even with negative logits, got {fake_prob.item()}"

    def test_normalization_formula_edge_cases(self):
        """
        InstructBLIPMathCheck (edge cases):
        Verify the formula handles extreme logit gaps without NaN/Inf, and
        that direction is correct:
          - Large no_logit  → P_fake near 1.0  (model says "No" = Fake)
          - Large yes_logit → P_fake near 0.0  (model says "Yes" = Real)
        """
        # No dominates → high fake probability, no NaN/Inf
        yes_logit = torch.tensor([-50.0])
        no_logit  = torch.tensor([50.0])
        fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
        assert not torch.isnan(fake_prob).any()
        assert not torch.isinf(fake_prob).any()
        assert fake_prob.item() > 0.99, "Large no_logit dominance should push P_fake near 1"

        # Yes dominates → image should lean Real
        yes_logit = torch.tensor([5.0])
        no_logit  = torch.tensor([0.1])
        fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
        assert fake_prob.item() < 0.5, "When yes_logit >> no_logit, image should lean Real"

        # No dominates → image should lean Fake
        yes_logit = torch.tensor([0.1])
        no_logit  = torch.tensor([5.0])
        fake_prob = normalize_to_fake_probability(yes_logit, no_logit)
        assert fake_prob.item() > 0.5, "When no_logit >> yes_logit, image should lean Fake"

    # -----------------------------------------------------------------------
    # Token ID extraction – de-duplication
    # -----------------------------------------------------------------------
    def test_yes_no_token_ids_no_duplicates(self, processor_and_model):
        """
        Verify that get_yes_no_token_ids successfully encodes Yes/No variants
        to valid, non-overlapping token IDs.

        De-duplication is required because the tokenizer may map "Yes" and
        " Yes" to the same ID — counting it twice would skew the MAX logit.
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)

        yes_ids_t, no_ids_t = get_yes_no_token_ids(proc.tokenizer, device)

        assert len(yes_ids_t) > 0, "Yes token IDs must not be empty"
        assert len(no_ids_t)  > 0, "No token IDs must not be empty"
        assert (yes_ids_t > 0).all(), "All Yes token IDs must be valid positive IDs"
        assert (no_ids_t  > 0).all(), "All No token IDs must be valid positive IDs"

        overlap = set(yes_ids_t.tolist()) & set(no_ids_t.tolist())
        assert len(overlap) == 0, \
            f"Yes and No token sets share IDs {overlap} — classification is undefined"

    def test_yes_no_overlap_raises(self, processor_and_model):
        """
        Verify that get_yes_no_token_ids fails loudly (raises ValueError)
        if Yes/No token IDs collide, rather than silently producing a
        corrupted P(Fake).
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)

        class _CollidingTokenizer:
            def encode(self, text, add_special_tokens=False):
                return [999]  # every token collides on the same ID

        with pytest.raises(ValueError, match="overlap"):
            get_yes_no_token_ids(_CollidingTokenizer(), device)

    # -----------------------------------------------------------------------
    # Logit extraction shape
    # -----------------------------------------------------------------------
    def test_logit_extraction_shape(self, processor_and_model, dummy_face_image):
        """
        Verify that output.scores[0] has shape (batch_size, vocab_size).

        The algorithm halts generation at the first token (max_new_tokens=1)
        and reads logits from outputs.scores[0] — the raw unnormalized scores
        over the full vocabulary before any sampling step.
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)
        dtype  = _get_dtype(mdl)

        inputs = proc(images=dummy_face_image, text="is this photo real?", return_tensors="pt")
        inputs = {k: (v.to(device, dtype=dtype) if k == 'pixel_values' else v.to(device))
                  for k, v in inputs.items()}

        with torch.no_grad():
            outputs = mdl.generate(
                **inputs,
                max_new_tokens=1,
                return_dict_in_generate=True,
                output_scores=True
            )

        assert hasattr(outputs, 'scores')
        assert len(outputs.scores) >= 1

        first_token_logits = outputs.scores[0]
        # Use the model's actual output dimension, not tokenizer.vocab_size.
        # HuggingFace often pads the LM head output to a multiple (e.g. 64)
        # for compute efficiency, so model_vocab_size >= tokenizer.vocab_size.
        model_vocab_size = mdl.language_model.lm_head.out_features
        assert first_token_logits.shape == (1, model_vocab_size), \
            f"Expected shape (1, {model_vocab_size}), got {first_token_logits.shape}"


# ===========================================================================
# 2. Data Pipeline Tests
# ===========================================================================

class TestDataPipeline:

    def test_eval_dataset_loads_image(self, processor_and_model, tmp_path):
        """
        Verify EvalDeepfakeDataset correctly loads an image and returns
        the required keys expected by the evaluation loop.
        """
        from torch.utils.data import Dataset

        proc, _ = processor_and_model
        img_path = str(tmp_path / "test_real.jpg")
        Image.new('RGB', (224, 224), color=(100, 150, 200)).save(img_path)

        df = pd.DataFrame({'image_path': [img_path], 'label': ['Yes']})

        class EvalDeepfakeDataset(Dataset):
            def __init__(self, data_frame, processor):
                self.data = data_frame
                self.processor = processor
            def __len__(self):
                return len(self.data)
            def __getitem__(self, idx):
                row = self.data.iloc[idx]
                image = Image.open(row['image_path']).convert('RGB')
                inputs = self.processor(
                    images=image, text="is this photo real?", return_tensors="pt"
                )
                inputs = {k: v.squeeze(0) for k, v in inputs.items()}
                inputs['text_label'] = row['label']
                return inputs

        dataset = EvalDeepfakeDataset(df, proc)
        assert len(dataset) == 1

        sample = dataset[0]
        assert 'pixel_values' in sample
        assert 'text_label'   in sample
        assert sample['text_label'] == 'Yes'

    def test_label_encoding(self):
        """
        Verify the label encoding logic:
        'No'  (Fake) → numerical label 1
        'Yes' (Real) → numerical label 0

        Matches the evaluation loop: "no" in label.lower() → 1 else 0.
        """
        labels   = ['Yes', 'No', 'yes', 'no']
        expected = [0, 1, 0, 1]
        encoded  = [1 if "no" in lbl.lower() else 0 for lbl in labels]
        assert encoded == expected, f"Label encoding mismatch: {encoded}"

    def test_grayscale_image_converted_to_rgb(self, processor_and_model, tmp_path):
        """
        Verify that grayscale images (common in some datasets) are correctly
        converted to RGB before processing. A grayscale image passed directly
        to the processor would produce wrong channel dimensions.
        """
        proc, _ = processor_and_model
        gray_path = str(tmp_path / "gray.jpg")
        Image.new('L', (224, 224), color=128).save(gray_path)

        image = Image.open(gray_path).convert('RGB')
        assert image.mode == 'RGB', "Grayscale image must be converted to RGB"

        inputs = proc(images=image, text="is this photo real?", return_tensors="pt")
        assert inputs['pixel_values'].shape[1] == 3, \
            "pixel_values must have 3 channels after grayscale→RGB conversion"

    def test_small_image_processed_without_error(self, processor_and_model):
        """
        Verify that unusually small images (e.g., 32x32) do not crash the
        processor. The processor should resize internally.
        """
        proc, _ = processor_and_model
        tiny_image = Image.new('RGB', (32, 32), color=(0, 0, 0))
        try:
            inputs = proc(images=tiny_image, text="is this photo real?", return_tensors="pt")
            assert 'pixel_values' in inputs
        except Exception as e:
            pytest.fail(f"Processor crashed on small image: {e}")


# ===========================================================================
# 3. System Integration Tests
# ===========================================================================

class TestSystemIntegration:

    # -----------------------------------------------------------------------
    # DetectRealSanity
    # -----------------------------------------------------------------------
    def test_detect_real_sanity(self, processor_and_model):
        """
        DetectRealSanity:
        Run the full probabilistic inference pipeline and verify P_fake is
        a valid float in [0, 1].

        Note: a white image is not a face. This test validates pipeline
        integrity (model loads, inference runs, score is valid), not
        model accuracy on real faces.
        """
        proc, mdl = processor_and_model
        image     = Image.new('RGB', (224, 224), color=(255, 255, 255))
        fake_prob = _run_probabilistic_inference(mdl, proc, image)

        assert 0.0 <= fake_prob <= 1.0, \
            f"P_fake must be in [0, 1], got {fake_prob}"

    # -----------------------------------------------------------------------
    # DetectFakeSanity
    # -----------------------------------------------------------------------
    def test_detect_fake_sanity(self, processor_and_model):
        """
        DetectFakeSanity:
        Run the full probabilistic inference pipeline on a blended image
        and verify P_fake is a valid float in [0, 1].

        Note: we do NOT assert P_fake > 0.5. Whether InstructBLIP can
        detect synthetic blending artifacts in zero-shot is the research
        question itself — encoding a threshold expectation here would turn
        an open hypothesis into a test constraint.
        """
        proc, mdl = processor_and_model

        # Simulate a blended image: two uniform color regions merged
        source  = np.full((224, 224, 3), fill_value=200, dtype=np.uint8)
        target  = np.full((224, 224, 3), fill_value=50,  dtype=np.uint8)
        mask    = np.zeros((224, 224, 1), dtype=np.float32)
        mask[60:160, 60:160, :] = 0.75
        blended = (source.astype(np.float32) * mask +
                   target.astype(np.float32) * (1.0 - mask)).astype(np.uint8)
        image   = Image.fromarray(blended)

        fake_prob = _run_probabilistic_inference(mdl, proc, image)

        assert 0.0 <= fake_prob <= 1.0, \
            f"P_fake must be in [0, 1] for blended image, got {fake_prob}"

    # -----------------------------------------------------------------------
    # Pipeline reproducibility (determinism)
    # -----------------------------------------------------------------------
    def test_pipeline_is_deterministic(self, processor_and_model):
        """
        Verify that running the inference pipeline twice on the same image
        produces the exact same P_fake score.

        Expected because:
          - model.eval() disables dropout
          - max_new_tokens=1 with output_scores=True extracts raw logits
            without any sampling step
        """
        proc, mdl = processor_and_model
        image     = Image.new('RGB', (224, 224), color=(150, 100, 80))

        prob1 = _run_probabilistic_inference(mdl, proc, image)
        prob2 = _run_probabilistic_inference(mdl, proc, image)

        assert prob1 == prob2, \
            f"Pipeline is non-deterministic: run1={prob1:.6f}, run2={prob2:.6f}"

    # -----------------------------------------------------------------------
    # Batch inference
    # -----------------------------------------------------------------------
    def test_batch_inference_produces_correct_count_and_range(self, processor_and_model):
        """
        Verify that batch inference (as used in the evaluation DataLoader)
        produces exactly one P_fake per input image, all in [0, 1], and
        that different images can produce different scores.

        Mirrors the evaluation loop:
          generate(max_new_tokens=1) → MAX over de-duped IDs → Softmax.
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)
        dtype  = _get_dtype(mdl)

        images = [
            Image.new('RGB', (224, 224), color=(255, 255, 255)),  # white
            Image.new('RGB', (224, 224), color=(0, 0, 0)),        # black
            Image.new('RGB', (224, 224), color=(120, 60, 200)),   # purple
        ]
        batch_size = len(images)

        inputs = proc(images=images, text=["is this photo real?"] * batch_size,
                      return_tensors="pt", padding=True)
        inputs = {k: (v.to(device, dtype=dtype) if k == 'pixel_values' else v.to(device))
                  for k, v in inputs.items()}

        yes_ids, no_ids = get_yes_no_token_ids(proc.tokenizer, device)

        with torch.no_grad():
            outputs = mdl.generate(
                **inputs,
                max_new_tokens=1,
                return_dict_in_generate=True,
                output_scores=True
            )

        first_token_logits = outputs.scores[0]  # (batch_size, vocab_size)
        assert first_token_logits.shape[0] == batch_size, \
            f"Expected {batch_size} rows of logits, got {first_token_logits.shape[0]}"

        yes_logit  = first_token_logits[:, yes_ids].max(dim=-1).values
        no_logit   = first_token_logits[:, no_ids].max(dim=-1).values
        fake_probs = normalize_to_fake_probability(yes_logit, no_logit)

        assert fake_probs.shape == (batch_size,), \
            f"Expected {batch_size} P_fake values, got shape {fake_probs.shape}"

        for i, p in enumerate(fake_probs.cpu().tolist()):
            assert 0.0 <= p <= 1.0, \
                f"P_fake for batch image {i} must be in [0, 1], got {p}"

    # -----------------------------------------------------------------------
    # ThresholdCheck
    # -----------------------------------------------------------------------
    def test_threshold_check(self):
        """
        ThresholdCheck:
        Verify that threshold-based classification returns the correct label.
        Boundary condition: P_fake >= 0.5 is classified as Fake.
        """
        threshold = 0.5

        def classify(prob, thresh):
            return "Fake" if prob >= thresh else "Real"

        assert classify(0.8,  threshold) == "Fake"
        assert classify(0.51, threshold) == "Fake"
        assert classify(0.50, threshold) == "Fake"   # boundary: >= is Fake
        assert classify(0.49, threshold) == "Real"
        assert classify(0.2,  threshold) == "Real"

    # -----------------------------------------------------------------------
    # AUC validity
    # -----------------------------------------------------------------------
    def test_auc_score_range(self):
        """
        Verify AUC computed from P_fake scores lies in [0, 1].
        """
        from sklearn.metrics import roc_auc_score

        # Perfect classifier
        labels = np.array([0, 0, 0, 1, 1, 1])
        probs  = np.array([0.1, 0.2, 0.15, 0.8, 0.9, 0.85])
        auc = roc_auc_score(labels, probs)
        assert 0.0 <= auc <= 1.0
        assert auc == 1.0, "Perfect separation should yield AUC = 1.0"

        # Random classifier
        np.random.seed(42)
        labels_rand = np.random.randint(0, 2, 100)
        probs_rand  = np.random.rand(100)
        auc_rand = roc_auc_score(labels_rand, probs_rand)
        assert 0.0 <= auc_rand <= 1.0

    # -----------------------------------------------------------------------
    # Embedding shape
    # -----------------------------------------------------------------------
    def test_embedding_extraction_shape(self, processor_and_model, dummy_face_image):
        """
        Verify that Q-Former embeddings have the expected shape
        (batch_size, embedding_dim) after mean pooling.

        This mirrors the embedding extraction in the evaluation loop used
        for t-SNE clustering analysis.
        """
        proc, mdl = processor_and_model
        device = _get_device(mdl)
        dtype  = _get_dtype(mdl)

        inputs = proc(
            images=dummy_face_image,
            text="is this photo real?",
            return_tensors="pt"
        )
        pixel_values           = inputs['pixel_values'].to(device, dtype=dtype)
        qformer_input_ids      = inputs['qformer_input_ids'].to(device)
        qformer_attention_mask = inputs['qformer_attention_mask'].to(device)

        with torch.no_grad():
            vision_outputs = mdl.vision_model(
                pixel_values,
                output_attentions=False,
                output_hidden_states=False,
                return_dict=True,
            )
            image_hidden_states = vision_outputs.last_hidden_state

            qformer_outputs = mdl.qformer(
                input_ids=qformer_input_ids,
                attention_mask=qformer_attention_mask,
                query_embeds=mdl.query_tokens.expand(pixel_values.shape[0], -1, -1),
                encoder_hidden_states=image_hidden_states,
            )

        embedding = qformer_outputs.last_hidden_state.mean(dim=1)

        assert embedding.ndim == 2, \
            f"Embedding must be 2-D (batch, dim), got {embedding.shape}"
        assert embedding.shape[0] == 1, "Batch size must be 1 for single-image test"
        assert embedding.shape[1] > 0,  "Embedding dimension must be positive"

Writing test_deepfake_detector.py


In [ ]:
!pytest test_deepfake_detector.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.15
collected 20 items                                                             

test_deepfake_detector.py::TestInstructBLIPModule::test_model_load PASSED [  5%]
test_deepfake_detector.py::TestInstructBLIPModule::test_image_preparation PASSED [ 10%]
test_deepfake_detector.py::TestInstructBLIPModule::test_get_text_output PASSED [ 15%]
test_deepfake_detector.py::TestInstructBLIPModule::test_normalization_formula_basic PASSED [ 20%]
test_deepfake_detector.py::TestInstructBLIPModule::test_normalization_handles_negative_logits PASSED [ 25%]
test_deepfake_detector.py::TestInstructBLIPModule::test_normalization_formula_edge_cases PASSED [ 30%]
test_deepfake_detector.py::TestInstructBLIPModule::test_yes_no_token_ids_no_duplicates PASSED 